# Quantitative Trading Framework
## Walk-Forward Optimization: Strategy Validation & Engine Evolution

**Asset:** BTC/USDT (Binance Perpetual) 
**Period:** January 2021 -- March 2026 (5.1 years, 179,979 bars at 15m) 
**Engine:** WFO v2.1 → v2.2 
**Date:** 3 March 2026

---

> **Note:** Strategy signal generation logic has been removed from this public repository to protect proprietary IP. The framework architecture, optimization engine, analytics dashboard, and all supporting infrastructure are fully included.

This notebook presents the full lifecycle of Walk-Forward Optimization applied to a session-level mean-reversion strategy on BTC/USDT. It documents:

1. The framework's risk-first philosophy and why standard backtesting fails
2. Initial WFO results across 6 adapter/timeframe combinations
3. Three statistical bug fixes and two architecture upgrades to the engine
4. Construction and validation of a live-faithful simulation adapter
5. Monte Carlo analysis comparing live performance vs WFO
6. Regime attribution and forward risk assessment

---
# Part I: Framework Philosophy
---

## The Problem

Most retail backtests are worthless. A strategy is optimized on historical data, produces an impressive equity curve, and then loses money in production. The root cause is almost always the same: **in-sample overfitting disguised as performance**.

A strategy trained on 2024 BTC price action will naturally "learn" patterns specific to that year — the ETF approval rally, the halving anticipation, the summer consolidation. These patterns won't repeat. A naive backtest that trains and tests on the same data reports high Sharpe ratios because it is literally grading its own homework.

This framework exists to eliminate that failure mode through four layers of defense:

1. **Walk-forward optimization** — Parameter tuning and performance evaluation happen on strictly separated data
2. **Monte Carlo simulation** — Trade sequences are reshuffled thousands of times to separate genuine edge from favorable sequencing
3. **Regime-conditional validation** — Performance is measured per market regime, exposing strategies that only work in one environment
4. **Stress testing under synthetic tail events** — Historical crash scenarios are scaled and applied to test whether the strategy survives conditions it has never encountered

## Why Walk-Forward Optimization

Static backtesting optimizes parameters on a dataset and then reports performance on that same dataset. Walk-forward optimization (WFO) does something fundamentally different: it repeatedly splits time-series data into a training window (in-sample) and a validation window (out-of-sample), optimizes on the training data, and evaluates only on the unseen validation window.

**Why this matters:**

- **Overfitting detection is built in.** The overfit ratio (IS performance / OOS performance) directly measures how much the strategy degrades on unseen data. A ratio near 1.0 means the parameters generalize; a ratio above 2.0 means the strategy memorized noise.
- **Parameters adapt to structural change.** Rolling windows allow the optimizer to discover that the best parameters for a trending market (Q4 2024) differ from those in a ranging market (Q3 2024).
- **Statistical significance is testable.** Each OOS window produces independent performance samples. With 30+ windows, Sharpe ratios can be tested for significance.

## Why Monte Carlo Simulation

A single backtest produces a single equity curve — one path through history. But that path is heavily influenced by trade ordering. A strategy with 55% win rate and 1.5:1 reward-to-risk can produce equity curves ranging from +40% to -15% simply by reshuffling the order of the same trades.

Monte Carlo simulation (10,000 iterations with optional block bootstrap) answers this by constructing the full probability distribution of outcomes.

| Metric | What It Measures |
|:-------|:-----------------|
| **Value at Risk (95%)** | The loss threshold exceeded only 5% of the time |
| **Conditional VaR (CVaR)** | Expected loss *given* that VaR is breached — captures tail severity |
| **Max Drawdown Distribution** | Percentile bands (p50, p75, p90, p95) for peak-to-trough decline |
| **Probability of Ruin** | Likelihood of account drawdown exceeding a catastrophic threshold |
| **Sharpe Ratio CI** | Confidence interval on risk-adjusted returns |
| **Win Rate CI** | Wilson binomial confidence interval |

**Why block bootstrap?** Standard Monte Carlo assumes trades are independent. In practice, crypto strategies exhibit serial correlation — winning streaks during trends, losing streaks during chop. Block bootstrap preserves these autocorrelation structures by reshuffling in chunks rather than individual trades.

**Three data source modes:**
- **WFO OOS Trades** (recommended) — Only out-of-sample trades from walk-forward validation
- **Real Trades** — Reshuffles actual PnLs from live VPS bots
- **Synthetic Backtest** — Single-pass backtest followed by reshuffling

## Why Regime Detection

A strategy that averages +0.3R per trade may be earning +0.8R in trending markets and -0.4R in ranging markets. Aggregate metrics hide this.

| Regime | Detection Criteria | Implication |
|:-------|:-------------------|:------------|
| **Trending (up/down)** | ADX > 30, EMA50/200 alignment | Momentum strategies have edge; mean-reversion strategies lose |
| **Ranging** | ADX <= 30 | Mean-reversion works; trend-following generates whipsaws |
| **Volatile** | ATR > 1.8x 20-bar average | Wider stops required; position sizing must decrease |

**Why this matters for capital allocation:** A portfolio running four strategies should not allocate equally to all four during a volatile regime if three of them have negative expectancy in volatility. Regime detection enables dynamic allocation.

## Stress Testing and Tail Risk

Backtests only cover market conditions that have already occurred. Stress testing asks: **what happens under conditions that haven't?**

| Scenario | Mechanics | Calibration |
|:---------|:----------|:------------|
| **Historical Crash Scaling** | Reproduces actual crash dynamics at 1x-3x magnitude | May 2021 (-53%), FTX Nov 2022 (-27%), Luna/3AC Jun 2022 (-37%) |
| **Volatility Spike** | ATR multiplied by 2x-5x for sustained periods | Captures environments where normal stop distances are insufficient |
| **Flash Crash** | -10% to -15% in 3 bars with 50-70% recovery | Tests whether the strategy is stopped out before the recovery |
| **Prolonged Drawdown** | 200-bar downward drift with dead-cat bounces | Simulates multi-week bear conditions |
| **V-Shaped Recovery** | Crash followed by mirror recovery | Tests whether the strategy re-enters after a drawdown |

A strategy that scores above 70 across all scenario types at 2x magnitude demonstrates genuine robustness.

---
# Part II: WFO Engine Architecture
---

## Walk-Forward Window Layout

Each WFO run executes 105 rolling windows across 5.1 years of 15-minute BTC data:

```
IS Train: 8000 bars (~83 days)     OOS Test: 1600 bars (~17 days)
├──────────────────────────────┤    ├────────────────┤
                                    ↓ step by 1600
                ├──────────────────────────────┤    ├────────────────┤
                                                    ↓ step by 1600
                                ├──────────────────────────────┤    ├────┤

→ 105 non-overlapping OOS windows spanning Jan 2021 – Mar 2026
```

Within each IS window, the optimizer evaluates 150 Sobol-sampled parameter combinations, scores them by expectancy, and applies the best to the OOS window. Every OOS trade is genuinely out-of-sample.

## Strategy Adapter Pattern

The engine is adapter-agnostic. Each strategy implements a lightweight interface that separates signal generation from trade simulation:

In [ ]:
# backtrader_framework/optimization/strategy_adapters/base_adapter.py

class StrategyAdapter(ABC):
    @abstractmethod
    def generate_signals(self, df, params, scan_start, scan_end) -> List[Signal]:
        """Stateless: produce trade signals from OHLCV + indicators."""
        ...

    def execute_signals(self, df, params, scan_start, scan_end,
                        costs, max_bars, ...) -> Optional[List[TradeResult]]:
        """Optional: stateful signal-to-trade processing.
        
        Returns None  → engine falls back to generate_signals() + simulate().
        Returns List  → engine uses these directly.
        
        Added in v2.2 to support adapters requiring sequential state
        (single-position constraint, re-entries). Fully backward-compatible."""
        return None

## Engine Implementation

The engine supports:

- **Timeframe-scaled windows** — `WFOConfig.for_timeframe()` automatically scales IS/OOS window sizes (a 15m chart has 16x more bars than a 4h chart for the same calendar period)
- **Regime detection** — Each window is classified as trending (up/down), ranging, or volatile based on ADX, ATR percentile, and EMA alignment
- **Grid search modes** — Full grid, random sampling (50-150 combinations), or Bayesian optimization via Optuna
- **Transaction cost modeling** — Every simulated trade deducts spread (0.01%), commission (0.055%), and slippage (0.01%), calibrated per asset
- **Combinatorial purged cross-validation** — Embargo zones between training and test sets prevent information leakage

---
# Part III: Initial WFO Results (Proxy Adapter)
---

## Adapter Variants Tested

**V2 (7 params):** Session sweep detection, killzone filtering, confidence scoring, fixed R:R. 
**V3 (9 params):** V2 + regime gating, confirmation bars, ATR trailing, wider TP targets.

Both adapters use a stateless `generate_signals()` model: each signal is independently simulated with bar-level SL/TP resolution.

## OOS Performance Matrix

```
                    V2 @ 1H     V2 @ 15M    V2 @ 5M     V3 @ 1H     V3 @ 15M    V3 @ 5M
                    --------    --------    --------    --------    --------    --------
OOS Trades          224         147         99          159         125         123
Win Rate            33.9%       26.5%       34.3%       39.0%       44.0%       35.0%
Mean R              -0.245      -0.487      -0.392      -0.320      -0.356      -0.597
Profit Factor       0.537       0.261       0.346       0.507       0.466       0.303
Max DD (R)          57.88       72.40       41.30       52.31       45.57       75.23
MC p(profitable)    0.01%       0.0%        0.0%        0.01%       0.0%        0.0%
```

**All six configurations produce deeply negative OOS expectancy.** Monte Carlo resampling (10,000 reshuffles) confirms zero chance of profitability under any trade ordering.

## V2 vs V3: Complexity Harmed Performance

V3's additional parameters (regime score weights, confirmation bar count, trailing multiplier) expanded the search space from 7 to 9 dimensions:

| Metric | V2 (best) | V3 (best) | Direction |
|:-------|:----------|:----------|:----------|
| Mean R | -0.245 (1H) | -0.320 (1H) | V2 wins |
| PBO | 43.4% (5M) | 55.8% (15M) | V2 wins |
| Param Stability | 0.377 (15M) | 0.000 (all) | V2 wins |
| TP2 Hit Rate | 26-34% | 0.8-2.4% | V2 wins |

V3's confirmation bar filter delayed entries until the reversal was exhausted, destroying the TP hit rate. The wider parameter space inflated PBO from ~45% to ~60%, confirming harmful overfitting.

**Verdict: V2 is strictly superior to V3, but neither is tradeable.**

---
# Part IV: Engine Bug Fixes & Statistical Improvements
---

Audit of the WFO engine against peer-reviewed methodology revealed **three bugs** and **two architecture gaps**. All were corrected before the live-faithful adapter work.

## Bug Fix 1: Deflated Sharpe Ratio

The DSR call was broken — wrong parameter names, missing skewness/kurtosis inputs. The function existed in `cpcv.py` but was called with incorrect arguments, causing it to silently return `None`.

In [ ]:
# BEFORE (broken — wrong param names, missing higher moments)
dsr = deflated_sharpe_ratio(
    observed_sharpe=sharpe,     # wrong: param is 'observed_sr'
    n_trades=len(r_vals),       # wrong: param is 'n_returns'
    n_trials=150,               # wrong: param is 'num_trials'
    returns=r_vals,             # nonexistent parameter
)
# Result: None — silently broken, no DSR computed

In [ ]:
# AFTER (corrected — matches cpcv.py signature)
import scipy.stats

dsr = deflated_sharpe_ratio(
    observed_sr=sharpe,
    num_trials=n_combos,
    n_returns=len(r_vals),
    skewness=float(scipy.stats.skew(r_vals)),
    kurtosis=float(scipy.stats.kurtosis(r_vals, fisher=False)),  # Raw kurtosis (3.0 = normal)
)
# Result: {'deflated_sr': -48.57, 'p_value': 1.0, 'is_significant': False}

**Impact:** DSR now produces corrected values (-42 to -76) that properly penalize multiple testing across 150 parameter combinations per window. The non-normality correction via skewness and excess kurtosis is material: BTC returns exhibit fat tails (kurtosis >> 3) that inflate naive Sharpe estimates.

The DSR formula adjusts the observed Sharpe by the expected maximum Sharpe under the null hypothesis of no edge:

$$E[\max(SR)] \approx \sqrt{2 \ln N} - \frac{\ln \pi + \ln \ln N}{2\sqrt{2 \ln N}}$$

where $N$ = number of parameter combinations tested.

## Bug Fix 2: Probability of Backtest Overfitting (PBO)

The `compute_pbo()` function existed in `cpcv.py` but was never called — dead code. The fix wired it into the compilation step using the score matrices already collected for parameter stability:

In [ ]:
# backtrader_framework/optimization/wfo_engine.py — _compile_results()

# Build score matrix: rows = windows, cols = param combos
score_matrix = np.full((n_windows, n_combos), np.nan)
for w_idx, combo_scores in enumerate(self._all_window_combo_scores):
    for params, score in combo_scores:
        key = tuple(sorted(params.items()))
        if key in combo_to_col:
            score_matrix[w_idx, combo_to_col[key]] = score

# Split into halves for combinatorial cross-validation
half = n_windows // 2
is_perf = score_matrix[:half]
oos_perf = score_matrix[half:]
pbo_result = CPCV.compute_pbo(is_perf, oos_perf)

**Impact:** PBO values range from 40-64% across configurations. Values above 50% indicate the optimizer is actively harmful — random parameter selection would outperform the "optimized" choice.

```
PBO Results:     V2 @ 1H    V2 @ 15M    V2 @ 5M    V3 @ 1H    V3 @ 15M    V3 @ 5M
                 50.0%      44.2%       43.4%      57.7%      55.8%       64.2%
```

V3's higher PBO (56-64%) is directly linked to its 9-parameter space: more knobs → more convincing IS noise patterns → worse OOS.

## Bug Fix 3: Minimum OOS Trade Enforcement

`min_total_oos_trades` was configured (default 15) but never checked:

In [ ]:
# Guard inserted before statistics compilation
if len(unique_oos_trades) < self.config.min_total_oos_trades:
    return {
        'error': f'Insufficient OOS trades: {len(unique_oos_trades)} '
                 f'< {self.config.min_total_oos_trades}',
        'n_windows': len(self.window_results),
        'param_history': self.param_history,
    }

**Impact:** Prevents misleading statistics from sparse configurations.

## New Feature: Parameter Stability Analysis

Measures whether optimal parameters sit on a broad plateau (robust) or an isolated spike (overfit). The stability ratio compares the best score to the mean score of its immediate neighbors in parameter space:

In [ ]:
# backtrader_framework/optimization/param_grid.py

def compute_stability_ratio(best_score, combo_scores, param_specs):
    """stability_ratio = mean(neighbor_scores) / best_score.
    ~1.0 = flat/stable optimum (robust).
    <0.5 = spike (fragile/overfit)."""
    
    best_params = max(combo_scores, key=lambda x: x[1])[0]
    neighbors = get_param_neighbors(best_params, param_specs)
    
    score_map = {tuple(sorted(p.items())): s for p, s in combo_scores}
    neighbor_scores = [score_map[key] for nb in neighbors 
                       if (key := tuple(sorted(nb.items()))) in score_map]
    
    if not neighbor_scores:
        return {'stability_ratio': 0.0, 'n_neighbors_found': 0}
    return {'stability_ratio': np.mean(neighbor_scores) / best_score,
            'n_neighbors_found': len(neighbor_scores)}

**Results across all configurations:**

```
                V2 @ 1H    V2 @ 15M   V2 @ 5M    V3 @ 1H    V3 @ 15M   V3 @ 5M    Live @ 15M
Stability       0.053      0.377      -0.135     0.000      0.000      0.000      0.114
Fragile Win.    98/105     85/105     69/106     105/105    98/105     92/106     85/105
```

V3's uniform **0.000 stability** (100% fragile windows) is the clearest signal that its 9-parameter space is pure noise-fitting.

## New Feature: HMM Regime Assessment

A 2-state Gaussian Hidden Markov Model identifies calm vs volatile regimes with **three layers of leakage prevention**:

1. HMM is fit ONLY on in-sample data (never sees OOS bars)
2. Feature standardization uses IS-computed mean/std (no OOS statistics)
3. OOS inference is forward-filter ONLY (no backward smoothing)

In [ ]:
# backtrader_framework/optimization/hmm_regime.py

class HMMRegimeAssessor:
    FEATURES = ['LogReturn', 'RealizedVol20']
    
    def fit_on_is(self, is_df):
        """Layer 1: HMM fit on IS data only."""
        X = is_df[self.FEATURES].values
        self.is_mean = np.mean(X, axis=0)      # Layer 2: IS-fitted standardization
        self.is_std = np.std(X, axis=0)
        self.hmm.fit((X - self.is_mean) / self.is_std)

    def filter_oos(self, oos_df):
        """Layer 3: Forward-filter only — no backward pass."""
        X_std = (oos_df[self.FEATURES].values - self.is_mean) / self.is_std
        return self.hmm.forward_filter(X_std)   # P(state_t | x_1:t)

In [ ]:
# The forward filter — causal inference, no look-ahead
# backtrader_framework/optimization/hmm_regime.py — GaussianHMM.forward_filter()

def forward_filter(self, X):
    """P(state_t | x_1:t) — each bar uses only past data."""
    log_B = self._log_emission(X)
    log_alpha = np.zeros((T, self.K))
    log_alpha[0] = np.log(self.pi) + log_B[0]
    log_alpha[0] -= _logsumexp(log_alpha[0])

    for t in range(1, T):
        for j in range(self.K):
            log_alpha[t, j] = (
                _logsumexp(log_alpha[t-1] + np.log(self.A[:, j]))  # Transition
                + log_B[t, j]                                       # Emission
            )
        log_alpha[t] -= _logsumexp(log_alpha[t])  # Normalize to probabilities

    return np.exp(log_alpha)  # (T, K) filtered state probabilities

**Results:** HMM successfully fitted on all 105 windows. BTC spent ~65% of the 5-year period in "calm" regime with mean position sizing multiplier of 0.748.

Graduated sizing tiers from the HMM:

| P(calm) | Position Size | Interpretation |
|:--------|:-------------|:---------------|
| ≥ 0.70 | 1.0x (full) | High-confidence calm regime |
| ≥ 0.55 | 0.7x (reduced) | Moderate calm probability |
| P(vol) ≥ 0.60 | 0.3x (defensive) | High-confidence volatile regime |
| Otherwise | 0.5x (uncertain) | Regime ambiguous |

---
# Part V: Live-Faithful Adapter
---

## The Gap Between Live and WFO

The live VPS bot showed positive performance over a 94-day operating period, while the WFO proxy adapter produced negative OOS expectancy over 5 years. Analysis identified **six structural gaps** between the live bot's execution and the WFO simulation:

| Feature | WFO Proxy | Live Adapter |
|:--------|:----------|:-------------|
| Position management | All signals independently | Single position at a time |
| Re-entry after stop | Not modeled | Configurable max attempts, cooldown, window |
| Trailing stop | Simple ATR trail after TP1 | Stepped, direction-specific multipliers |
| Initial buffer | None | Virtual wider SL for initial bars |
| Time exit | Timeout at max_bars | Close after configurable duration if not in profit |
| Breakeven trigger | Move to entry at TP1 | Move to BE at configurable R threshold |

## Solution Architecture

Two components were built to close the gap:

1. **`TradeSimulator.simulate_v2()`** — Enhanced bar-by-bar simulation with all six execution features, controlled via signal metadata
2. **`LiquidityRaidLiveAdapter.execute_signals()`** — Sequential signal processing with single-position constraint and re-entry mechanism

In [ ]:
# backtrader_framework/optimization/simulator.py — simulate_v2()
# Enhanced bar loop with 6-priority execution model

@staticmethod
def simulate_v2(signal, df, costs, max_bars=168, ...):
    """V2 simulator — replicates live bot execution logic."""
    meta = signal.get('metadata', {})
    
    # All parameters driven by metadata (adapter controls behavior)
    buffer_bars     = meta.get('initial_buffer_bars', 0)
    buffer_mult     = meta.get('initial_buffer_mult', 1.35)
    be_trigger_r    = meta.get('breakeven_trigger_r', 0.5)
    time_exit_bars  = meta.get('time_exit_bars', 0)
    
    # Direction-specific trailing
    is_long = direction == 'LONG'
    trail_atr_mult = meta.get(
        'trail_atr_mult_long' if is_long else 'trail_atr_mult_short', 0
    )
    trail_step_atr = meta.get(
        'trail_step_atr_long' if is_long else 'trail_step_atr_short', 0
    )

    for i in range(idx + 1, end_bar):
        # Priority 1: Initial buffer check (first N bars)
        # Priority 2: Breakeven trigger (one-time ratchet)
        # Priority 3: Stepped trailing (direction-specific)
        # Priority 4: SL check (conservative: SL before TP)
        # Priority 5: TP check
        # Priority 6: Time exit (if not in profit after N bars)
        ...

In [ ]:
# backtrader_framework/optimization/strategy_adapters/liquidity_raid_live_adapter.py
# Sequential signal processing with state

class LiquidityRaidLiveAdapter(StrategyAdapter):
    
    def execute_signals(self, df, params, scan_start, scan_end,
                        costs, max_bars, window_id, is_oos, regime):
        signals = self.generate_signals(df, params, scan_start, scan_end)
        signals.sort(key=lambda s: s.idx)  # Chronological
        
        trades = []
        position_exit_bar = -1
        stop_outs = []
        
        # Pass 1: Primary signals with single-position constraint
        for sig in signals:
            if sig.idx < position_exit_bar:  # Position still open
                continue                      # Skip — single position only
            
            trade = TradeSimulator.simulate_v2(sig, df, costs, ...)
            if trade:
                trades.append(trade)
                position_exit_bar = sig.idx + trade.bars_held
                if trade.outcome == 'loss':
                    stop_outs.append(sig)  # Track for re-entry
        
        # Pass 2: Re-entry attempts after stop-outs
        # (scans for confirming candle within cooldown/window constraints)
        ...
        
        return trades

## Timeframe-Aware Constants

Duration-based parameters scale automatically with the data timeframe:

```python
def _bars_for_hours(hours, timeframe):
    tf_minutes = {'5m': 5, '15m': 15, '1h': 60, ...}
    return max(1, int(hours * 60 / tf_minutes[timeframe]))
```

| Constant | 5m | 15m | 1h |
|:---------|:---|:----|:---|
| Time exit (6h) | 72 bars | 24 bars | 6 bars |
| Initial buffer (1h) | 12 bars | 4 bars | 1 bar |
| Re-entry cooldown (1h) | 12 bars | 4 bars | 1 bar |
| Re-entry window (2.5h) | 30 bars | 10 bars | 3 bars |

## Live Adapter WFO Results

The live adapter was run through the same 105-window WFO on BTC 15m (150 param combos, 822 seconds):

```
Metric                   Proxy (V2)      Live Adapter
─────────────────────────────────────────────────────
OOS Trades               147             153
Win Rate                 26.5%           51.6%
DSR                      -48.6           -75.5
PBO                      44.2%           40.4%
Param Stability          0.377           0.114
```

**Key diagnostic:** The win rate jumped from 26.5% → 51.6%, confirming that single-position constraint, breakeven trigger, and initial buffer are genuine structural features. The PBO improved from 44.2% → 40.4%, indicating less overfitting per window.

However, 5-year OOS performance remains negative because the strategy's natural habitat (low-ADX ranging periods) constitutes only ~35% of BTC's history. The bar-level OHLC simulation also cannot fully resolve sub-bar execution advantages the live bot has.

---
# Part VI: Monte Carlo Analysis
---

## Live Bot vs WFO: 10,000 Bootstrap Reshuffles

Both simulations use 10,000 bootstrap reshuffles of the actual trade R-multiples.

```
Metric                          Live Bot (94d)     WFO Proxy (5yr)
─────────────────────────────────────────────────────────────────────
Trades                          94                 147
P(profitable)                   96.0%              0.0%

Final R Distribution:
  5th percentile                +1.14R             -87.44R
  Median                        +22.28R            -71.48R
  95th percentile               +43.72R            -55.02R

Max Drawdown Distribution:
  Median                        7.48R              71.77R
  95th percentile               14.28R             87.45R
```

**The gap is in the trade distributions, not in luck.** Reshuffling trade order 10,000 times cannot transform the WFO's negative mean distribution into profitability. Conversely, the live distribution is profitable in 96% of reshuffles — even the unluckiest 5% of scenarios end positive (+1.14R).

This confirms the live bot operates in a fundamentally different regime and/or benefits from execution features the proxy adapter cannot capture.

---
# Part VII: Regime Attribution & Risk Assessment
---

## Current Market State (as of 3 March 2026)

```
HMM:        CALM (P=0.987 on 15M, transitioning from 80% volatile last 14d)
ADX (1H):   14.6 (ranging, no directional trend)
Price:      $66,792, below all major EMAs (50/100/200)
30d change: -26.8%
```

The strategy's natural habitat is **low-ADX ranging markets** where session sweeps mean-revert. The current environment (post-ATH drawdown, ADX < 20) is optimal for the signal.

## Profitability Attribution

```
Source                      Contribution    Confidence
───────────────────────────────────────────────────────
Favorable regime            ~40%            High — ADX < 20, confirmed by HMM
Execution features          ~30%            High — WR jump 26% → 52% validates
(re-entry, buffer, trail)
Variance / small sample     ~30%            Moderate — 94 trades, CI includes zero
```

## Risk Indicators to Monitor

| Indicator | Threshold | Action |
|:----------|:----------|:-------|
| ADX (1H) | > 25 | Exit favorable regime, reduce exposure |
| Consecutive losses | > 5 | Potential regime shift |
| Win rate (20-trade rolling) | < 40% | Strategy degradation |
| Re-entry success rate | Declining | Market structure change |

---
# Part VIII: Engine Quality Improvements Summary
---

| Component | Before (v2.1) | After (v2.2) | Diagnostic Value |
|:----------|:--------------|:-------------|:-----------------|
| Deflated Sharpe | `None` (broken call) | -42 to -76 (corrected) | Quantifies multiple-testing penalty across 150 param combos |
| PBO | Dead code (never called) | 40-64% (wired) | Identifies when optimizer is harmful (>50% = worse than random) |
| Min OOS trades | Configured, not enforced | Guard with early return | Prevents misleading statistics from sparse windows |
| Param Stability | Not measured | 0.00 to 0.38 | Exposes fragile optima (V3 = 100% fragile) |
| HMM Regime | Not available | 2-state, 3-layer leakage prevention | Graduated sizing by regime probability |
| Simulator | Bar-level SL/TP only | +buffer, +breakeven, +stepped trail, +time exit | Matches live bot execution |
| Adapter interface | Stateless only | +optional `execute_signals()` | Supports sequential state |

---
# Part IX: Architecture & Infrastructure
---

## Repository Structure

```
backtrader_framework/
  strategies/         Strategy base class and adapter pattern
  indicators/         FVG detector, liquidity sweep detector, session tracker
  optimization/       WFO engine, Monte Carlo, SHAP analysis, Bayesian tuning,
                      portfolio optimization, stress testing, regime detection,
                      drawdown analysis, CPCV, cross-asset robustness
  data/               DuckDB OHLCV manager, data feeds, validation
  analyzers/          Sortino, Calmar, Omega ratio analyzers
  config/             Sessions, kill zones, indicator defaults

dashboard/            16-page Streamlit analytics application
  pages/              Overview, Strategy Deep Dive, Trade Journal, Equity Curves,
                      Session Analysis, Monthly Performance, ML Training,
                      Monte Carlo, Deploy, Portfolio, Meta Strategy, SHAP,
                      Bayesian Tuning, Stress Testing, Cross-Asset Robustness
  components/         Plotly charts, KPI cards, filters
  data/               VPS sync, data loading, schema normalization
```

## Portfolio Construction

| Method | Approach |
|:-------|:---------|
| **Risk Parity** | Equal risk contribution — each strategy contributes the same marginal portfolio volatility |
| **Mean-Variance (Markowitz)** | Maximum Sharpe allocation with Ledoit-Wolf shrinkage for covariance stability |
| **Kelly Criterion** | Optimal geometric growth sizing, fractional (0.25x) for practical drawdown limits |
| **Max Sharpe** | Maximize portfolio-level risk-adjusted returns subject to allocation constraints |

## Data Pipeline

- **DuckDB** — Embedded analytical database for local OHLCV storage (BTC/ETH at 15m/1h/4h)
- **Binance REST API** — Real-time market data (public endpoints)
- **VPS sync** — SSH-based retrieval of live trade databases from production servers
- **Schema normalization** — Unified trade format across heterogeneous bot database schemas

## Analytics Dashboard

A 16-page Streamlit application for strategy analysis, optimization, and monitoring:

| Page | Purpose |
|:-----|:--------|
| **Overview** | Multi-strategy KPI dashboard with real-time VPS sync |
| **Strategy Explainer** | Signal generation rules, parameter space, confidence scoring |
| **Strategy Deep Dive** | WFO results with regime-specific breakdowns |
| **Trade Journal** | Filterable trade log with MFE/MAE analysis |
| **Equity Curves** | Multi-strategy equity with changelog overlays |
| **Session Analysis** | Performance by trading session (Asia/London/NY) |
| **Monthly Performance** | Calendar heatmaps and monthly P&L tracking |
| **ML Training** | Feature engineering and model training pipeline |
| **Monte Carlo** | Probability distributions, VaR/CVaR, equity fan charts |
| **Deploy** | Bot deployment and service management |
| **Portfolio** | Multi-strategy allocation and correlation analysis |
| **Meta Strategy** | Dynamic strategy selection by market regime |
| **SHAP Analysis** | Feature importance and model interpretability |
| **Bayesian Tuning** | Optuna hyperparameter optimization |
| **Stress Testing** | Synthetic scenarios and survival scoring |
| **Cross-Asset Robustness** | Walk-forward validation across BTC/ETH/NQ |

## Tech Stack

| Category | Tools |
|:---------|:------|
| Language | Python 3.9+ |
| Backtesting | Backtrader, custom vectorized engine |
| Data | DuckDB, SQLite, pandas, NumPy |
| Optimization | Optuna (Bayesian), SciPy, custom WFO engine |
| ML | scikit-learn, XGBoost, SHAP |
| Dashboard | Streamlit, Plotly |
| APIs | Binance REST/WebSocket |

---
# Part X: Conclusions
---

1. **The core sweep-reversal signal does not constitute a durable 5-year edge in BTC.** This conclusion is consistent across all 7 WFO configurations, two adapter architectures, and three statistical tests (DSR, PBO, MC). The signal works in ranging regimes (~35% of history) but fails in trending regimes (~65%).

2. **The live bot's execution features are genuine improvements.** The WR jump from 26.5% → 51.6% when replicating single-position, re-entry, and breakeven logic is not overfitting — it's a structural change in trade selection.

3. **Live profitability is primarily regime-dependent.** The track record is too short (CI includes zero) and too regime-specific (ADX < 20, post-drawdown ranging) to extrapolate. Continued operation requires disciplined sizing and clear regime-based exit criteria.

4. **V3's added complexity was harmful.** Parameter stability of 0.000 and PBO of 56-64% confirm the 9-parameter space produces pure noise-fitting. Features should be validated against overfitting metrics, not just OOS expectancy.

5. **The engine's statistical corrections are now functioning.** DSR, PBO, parameter stability, and HMM regime assessment provide a robust diagnostic framework. Any new strategy variant should achieve DSR > 0, PBO < 25%, and stability > 0.5 before deployment consideration.

## Project Status

This framework powers live trading bots deployed on a VPS running 24/7. Four strategy families (8 bots) run continuously on BTC and ETH with automated Telegram reporting for trade alerts, daily performance summaries, and system health monitoring. The optimization and analytics infrastructure shown here is the same tooling used for strategy development, validation, and monitoring in production.

---

*Report generated from WFO Engine v2.2 results, VPS database sync (3 Mar 2026), and 10,000-run Monte Carlo analysis.*